### Homework 03 -  Evaluation of branching ratios
Igor Di Tota

In this exercise, I will evaluate the branching ratios of the $\omega(782)$ meson.
Three dominant decay channels are considered:
1. $\omega \to \pi^+\pi^-\pi^0$
2. $\omega \to \pi^0\gamma$
3. $\omega \to \pi^+\pi^-$

From $N = 4117$ measured decays, $N_1 = 3714$ were classified as channel 1, $N_2 = 338$ as channel 2 and $N_3 = 65$ as channel 3.

Each decay is a single trial whose outcome is one of the three channels, with probabilities $p_1, p_2, p_3$ (the branching ratios we are interested in).
Since we assume that every decay ends up in one of these three channels, the probabilities must satisfy the normalization condition
$$
p_1 + p_2 + p_3 = 1 .
$$


Because there are more than two outcomes, the joint probability to observe $N_1, N_2, N_3$ decays in the three channels out of $N$ independent decays is given by the **multinomial distribution**, the generalization of the binomial distribution:
$$
P(N_1, N_2, N_3) = \frac{N!}{N_1!\, N_2!\, N_3!}\, p_1^{N_1}\, p_2^{N_2}\, p_3^{N_3}, \qquad N_1 + N_2 + N_3 = N .
$$

As described in the PDG review, we can regard channel $i$ as "success" and all the other channels as "failure".
Then each $N_i$ individually follows a **binomial distribution** with $N$ tries, success probability $p_i$ and failure probability $q_i = 1 - p_i$:
$$
P(N_i) = \binom{N}{N_i}\, p_i^{N_i}\, q_i^{\,N - N_i} .
$$

Since the counts $N_i$ come from a multinomial distribution, the expected value and the variance of each $N_i$ are those of the binomial distribution:
$$
\mathbb{E}(N_i) = p_i N,
\qquad
\mathbb{V}(N_i) = p_i (1 - p_i) N .
$$

However, the $N_i$ are not independent: since $N$ is fixed, if one channel has more decays the others must have fewer.


In [19]:
import numpy as np

N_i = np.array([3714, 338, 65])
N = 4117

### Branching ratios and their uncertainties

As in the efficiency example from the lecture, the branching ratio of channel $i$ is estimated as the fraction of decays observed in that channel:
$$
\hat{p}_i = \frac{N_i}{N} .
$$

The variance of $N_i$ is given by the binomial distribution, $\mathbb{V}(N_i) = p_i(1-p_i)N$ (for fixed $N$).
Since $N$ is a constant and the variance is quadratic, $\mathbb{V}(aX) = a^2\,\mathbb{V}(X)$, the factor $1/N$ comes out squared.
So the expected variance of $\hat{p}_i$,  our estimate of $p_i$, is
$$
\sigma^2_{p_i} = \mathbb{V}\!\left(\frac{N_i}{N}\right) = \frac{\mathbb{V}(N_i)}{N^2} = \frac{p_i(1-p_i)\,N}{N^2} = \frac{\hat{p}_i\,(1-\hat{p}_i)}{N}.
$$

In [20]:
p = N_i / N
sigma_p = np.sqrt(p * (1 - p) / N)

for i in range(3):
    print(f"Channel {i+1}:  BR = {p[i]:.4f} +/- {sigma_p[i]:.4f}")

print(f"\nSum of the branching ratios: {p.sum():.4f}")


Channel 1:  BR = 0.9021 +/- 0.0046
Channel 2:  BR = 0.0821 +/- 0.0043
Channel 3:  BR = 0.0158 +/- 0.0019

Sum of the branching ratios: 1.0000


### Poisson vs binomial approach

As discussed in the lecture (slides 34-35), compute the uncertainty of $\hat{p}_i = N_i / N$ by treating $N_i$ and $N$ as two independent variables is a common mistake:
$$
\sigma^2_{N_i} = N_i, \qquad \sigma^2_{N} = N .
$$
Propagating the uncertainties for independent variables,definition given by $\sigma_f^2 = \sum_k \left(\frac{\partial f}{\partial x_k}\right)^2 \sigma_{x_k}^2$, with $\hat{p}_i = N_i / N$:
$$
\frac{\partial \hat{p}_i}{\partial N_i} = \frac{1}{N}, \qquad \frac{\partial \hat{p}_i}{\partial N} = -\frac{N_i}{N^2},
$$
so that
$$
\sigma^2_{p_i} = \left(\frac{1}{N}\right)^2 N_i + \left(\frac{N_i}{N^2}\right)^2 N
= \frac{N_i}{N^2} + \frac{N_i^2}{N^3}
= \frac{1}{N}\left(\frac{N_i}{N} + \frac{N_i^2}{N^2}\right)
= \frac{\hat{p}_i\,(1+\hat{p}_i)}{N} \qquad \textbf{\color{red}WRONG}
$$
This result is wrong because $N_i$ and $N$ are not independent: $N_i$ is always a subset of $N$.

The correct approach is to write the branching ratio as
$$
\hat{p}_i = \frac{N_i}{N_i + M_i}, \qquad M_i = N - N_i ,
$$
where $M_i$ is the number of decays in all the other channels.
Now $N_i$ and $M_i$ are two independent variables with Poisson uncertainty, $\sigma^2_{N_i} = N_i$ and $\sigma^2_{M_i} = M_i$.
The derivatives are
$$
\frac{\partial \hat{p}_i}{\partial N_i} = \frac{M_i}{(N_i + M_i)^2}, \qquad \frac{\partial \hat{p}_i}{\partial M_i} = -\frac{N_i}{(N_i + M_i)^2},
$$
so that, using $N_i + M_i = N$,
$$
\sigma^2_{p_i} = \left(\frac{M_i}{N^2}\right)^2 N_i + \left(\frac{N_i}{N^2}\right)^2 M_i
= \frac{N_i\, M_i\,(M_i + N_i)}{N^4}
= \frac{N_i\, M_i}{N^3}
= \frac{1}{N}\,\frac{N_i}{N}\,\frac{M_i}{N}
= \frac{\hat{p}_i\,(1-\hat{p}_i)}{N},
$$
which is the same result obtained from the binomial distribution.


In [21]:
# Using the above theory, numerically we have:
M_i = N - N_i

sigma_binom = sigma_p
sigma_wrong = np.sqrt((1 / N)**2 * N_i + (N_i / N**2)**2 * N)
sigma_poisson = np.sqrt((M_i / N**2)**2 * N_i + (N_i / N**2)**2 * M_i)

print(f"{'Channel':<8}{'BR':>10}{'binomial':>12}{'Poisson (N_i, M_i)':>22}{'WRONG (N_i, N)':>18}")
for i in range(3):
    print(f"{i+1:<8}{p[i]:>10.4f}{sigma_binom[i]:>12.4f}{sigma_poisson[i]:>22.4f}{sigma_wrong[i]:>18.4f}")


Channel         BR    binomial    Poisson (N_i, M_i)    WRONG (N_i, N)
1           0.9021      0.0046                0.0046            0.0204
2           0.0821      0.0043                0.0043            0.0046
3           0.0158      0.0019                0.0019            0.0020


The binomial approach and the correct Poisson approach give exactly the same uncertainties.

The wrong approach, which treats $N_i$ and $N$ as independent, overestimates the uncertainty.

In [22]:
channel_names = ["ω → π+ π- π0", "ω → π0 γ", "ω → π+ π-"]

print(f"{'Channel':<12}{'BR':>10}{'sigma':>10}{'rel. unc.':>12}")
for i in range(3):
    print(f"{channel_names[i]:<12}{p[i]:>10.4f}{sigma_p[i]:>10.4f}{sigma_p[i]/p[i]*100:>11.1f}%")


Channel             BR     sigma   rel. unc.
ω → π+ π- π0    0.9021    0.0046        0.5%
ω → π0 γ        0.0821    0.0043        5.2%
ω → π+ π-       0.0158    0.0019       12.3%
